# AEGIS — Large-Scale Public Dataset Benchmark

Interactive companion to `scripts/bench_aegis.py` for benchmarking AEGIS on the larger public datasets registered in [`datasets.yaml`](datasets.yaml).

**Use this notebook when**
- you need an interactive download flow (Kaggle credentials, TCIA collection picker, manual NIH login)
- you want the results plotted alongside the run, not just dumped to CSV

**Don't use this notebook for**
- the in-repo `pydicom_samples` smoke tier — run `python scripts/prepare_pydicom_samples.py` plus `scripts/bench_aegis.py` directly; it's faster and CI-friendly.

Everything here is config-driven by [`datasets.yaml`](datasets.yaml). Add an entry to that file and it becomes selectable below — no notebook edits required.

## 1. Setup — clone repo (if needed) and load the registry

This cell figures out where the repo lives and `chdir`s in. Three paths:

1. **Already inside a checkout** (running from a cloned repo locally, or from a path that contains `.git`): the cell just walks up to the repo root. No clone, no install.
2. **Hosted notebook with no checkout** (Google Colab, Kaggle, SageMaker, Binder, etc.): the cell clones the repo into a sensible per-platform default — `/content/aegis` on Colab, `/kaggle/working/aegis` on Kaggle, `<cwd>/aegis` everywhere else — then installs `monai_aegis` editable plus the notebook's own deps.
3. **Anywhere else** (custom remote box, container, friend's laptop): same as case 2, into `<cwd>/aegis`.

Override the defaults by exporting either of these before running:

- `AEGIS_REPO_URL` — clone source (default: the public GitHub URL).
- `AEGIS_REPO_DIR` — destination directory (overrides all platform defaults above).

The clone step is idempotent — if the destination already has a `.git`, it `git pull`s instead of re-cloning.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any, Dict, Optional


# ---------------------------------------------------------------------------
# Bootstrap — make sure the repo is on disk and we're cd'd into it.
# Skipped on a local checkout (where the repo root is already findable).
# ---------------------------------------------------------------------------
REPO_URL = os.environ.get('AEGIS_REPO_URL', 'https://github.com/lakshmi-mahabaleshwara/aegis.git')
ON_COLAB = 'google.colab' in sys.modules
ON_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or Path('/kaggle').is_dir()


def default_repo_dir() -> Path:
    """Pick a writable clone destination based on the host environment.

    The user can always override via the AEGIS_REPO_DIR env var.
    """
    if 'AEGIS_REPO_DIR' in os.environ:
        return Path(os.environ['AEGIS_REPO_DIR'])
    if ON_COLAB:
        return Path('/content/aegis')
    if ON_KAGGLE:
        return Path('/kaggle/working/aegis')
    # Generic fallback — sibling directory of wherever the notebook is launched.
    return Path.cwd() / 'aegis'


# Files that mark the repo root. Walking up from a notebook should hit
# one of these on a real checkout — `.git` is the canonical signal,
# `monai_aegis/pyproject.toml` is the fallback for shallow extractions.
ROOT_MARKERS = ('.git', 'monai_aegis/pyproject.toml')


def find_repo_root(start: Path) -> Optional[Path]:
    for candidate in [start, *start.parents]:
        if any((candidate / m).exists() for m in ROOT_MARKERS):
            return candidate
    return None


def bootstrap_repo() -> Path:
    # Already inside a checkout? Use it.
    found = find_repo_root(Path.cwd().resolve())
    if found is not None:
        return found

    # Not in a checkout — clone (or pull) into the resolved destination.
    repo_dir = default_repo_dir().resolve()
    if repo_dir.exists() and (repo_dir / '.git').exists():
        print(f'Updating existing checkout at {repo_dir} ...')
        subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
    else:
        repo_dir.parent.mkdir(parents=True, exist_ok=True)
        print(f'Cloning {REPO_URL} -> {repo_dir} ...')
        subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)

    os.chdir(repo_dir)

    # Install AEGIS + the few extras the notebook itself needs.
    print('Installing monai_aegis (editable) and notebook deps ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'monai_aegis/'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'pandas', 'matplotlib'], check=True)
    return repo_dir


REPO_ROOT = bootstrap_repo()
REGISTRY_PATH = REPO_ROOT / 'tests/benchmark/public_dataset/datasets.yaml'

import yaml  # imported after bootstrap in case PyYAML had to be installed

with open(REGISTRY_PATH) as f:
    REGISTRY: Dict[str, Any] = yaml.safe_load(f)

DATASETS: Dict[str, Dict[str, Any]] = REGISTRY['datasets']
BENCH_DEFAULTS: Dict[str, Any] = REGISTRY.get('benchmark_defaults', {})

print(f'\nRepo root : {REPO_ROOT}')
print(f'Host      : ' + ('Colab' if ON_COLAB else 'Kaggle' if ON_KAGGLE else 'local/other'))
print(f'Registry  : {REGISTRY_PATH.relative_to(REPO_ROOT)}')
print(f'Datasets  : {list(DATASETS)}')

## 2. Available datasets

Renders the registry as a table so you can pick one. The columns come straight from `datasets.yaml` — to surface a new field, add it to the YAML.

In [ ]:
import pandas as pd

rows = []
for key, entry in DATASETS.items():
    rows.append({
        'key': key,
        'source': (entry.get('source') or {}).get('type', ''),
        'modalities': ', '.join(entry.get('modalities', [])),
        'approx_size': entry.get('approx_size', ''),
        'license': entry.get('license', ''),
        'description': entry.get('description', ''),
    })

pd.DataFrame(rows).set_index('key')

## 3. Pick a dataset

Set `DATASET_KEY` to one of the keys above. Everything below this cell reads from the resolved entry — switch dataset by re-running this cell.

In [ ]:
DATASET_KEY = 'rsna_pneumonia'  # <-- edit me

if DATASET_KEY not in DATASETS:
    raise ValueError(f'{DATASET_KEY!r} not in registry. Available: {list(DATASETS)}')

ENTRY = DATASETS[DATASET_KEY]
TARGET_DIR = (REPO_ROOT / ENTRY['target_dir']).resolve()
SOURCE_TYPE = ENTRY['source']['type']

print(f'Selected     : {DATASET_KEY}')
print(f'Source type  : {SOURCE_TYPE}')
print(f'Target dir   : {TARGET_DIR}')
print(f'License      : {ENTRY.get("license", "")}')
print(f'Approx size  : {ENTRY.get("approx_size", "")}')
print(f'Source URL   : {ENTRY["source"].get("url", "(none)")}')

## 4. Prepare the dataset

Each source type has its own preparer. The dispatch table below makes it easy to add new ones — register a function under a new key and add the matching `source.type` to `datasets.yaml`.

| `source.type` | Preparer | Auth required |
|---------------|----------|---------------|
| `pydicom_builtin` | delegates to `scripts/prepare_pydicom_samples.py` | none |
| `kaggle` | `kaggle datasets download` / `kaggle competitions download` | Kaggle API token |
| `tcia` | prints `tcia_utils` snippet (manual install) | none for public collections |
| `http` | prints download URL — files saved manually to `target_dir` | varies |

If the preparer cannot run unattended (auth, ToS click-through), it prints the exact next step and exits — no silent half-downloads.

In [ ]:
def prep_pydicom_builtin(entry: Dict[str, Any], target: Path) -> None:
    cmd = [
        sys.executable,
        str(REPO_ROOT / 'scripts/prepare_pydicom_samples.py'),
        '--registry', str(REGISTRY_PATH),
        '--dataset', DATASET_KEY,
        '--target', str(target),
    ]
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)


def prep_kaggle(entry: Dict[str, Any], target: Path) -> None:
    if shutil.which('kaggle') is None:
        print('Install the Kaggle CLI first:  pip install kaggle')
        print('Then place your API token at  ~/.kaggle/kaggle.json  (chmod 600).')
        return
    target.mkdir(parents=True, exist_ok=True)
    slug = entry['source']['dataset']
    is_competition = '/' not in slug  # Kaggle competitions have no `owner/` prefix
    sub, flag = ('competitions', '-c') if is_competition else ('datasets', '-d')
    cmd = ['kaggle', sub, 'download', flag, slug, '-p', str(target), '--unzip']
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)


def prep_tcia(entry: Dict[str, Any], target: Path) -> None:
    src = entry['source']
    print('TCIA download is not bundled — install tcia_utils and run:')
    print()
    print('    pip install tcia_utils')
    print('    from tcia_utils import nbia')
    print(f'    series = nbia.getSeries(collection={src["collection"]!r})')
    print(f'    nbia.downloadSeries(series, path={str(target)!r})')
    print()
    print(f'Reference: {src.get("url", "https://www.cancerimagingarchive.net/")}')


def prep_http(entry: Dict[str, Any], target: Path) -> None:
    src = entry['source']
    print('This dataset requires manual download (ToS click-through / Box auth):')
    print(f'    URL    : {src.get("url")}')
    print(f'    Save to: {target}')


PREPARERS = {
    'pydicom_builtin': prep_pydicom_builtin,
    'kaggle': prep_kaggle,
    'tcia': prep_tcia,
    'http': prep_http,
}

In [ ]:
preparer = PREPARERS.get(SOURCE_TYPE)
if preparer is None:
    raise NotImplementedError(f'No preparer registered for source.type={SOURCE_TYPE!r}')

preparer(ENTRY, TARGET_DIR)

n_files = sum(1 for _ in TARGET_DIR.rglob('*') if _.is_file()) if TARGET_DIR.exists() else 0
print(f'\nFiles in {TARGET_DIR.relative_to(REPO_ROOT)}: {n_files}')

## 5. Run the AEGIS benchmark

Calls `scripts/bench_aegis.py` as a subprocess so the notebook stays decoupled from the script's internals — any flag the CLI grows is automatically usable here. Defaults come from `benchmark_defaults` in `datasets.yaml`; per-dataset entries can add a `benchmark` block to override them.

In [ ]:
from datetime import datetime

bench_settings = {**BENCH_DEFAULTS, **(ENTRY.get('benchmark') or {})}
config_path = REPO_ROOT / bench_settings['config']
output_root = REPO_ROOT / bench_settings['output'] / DATASET_KEY
mode = bench_settings.get('mode', 'single')

output_root.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(REPO_ROOT / 'scripts/bench_aegis.py'),
    '--input', str(TARGET_DIR),
    '--config', str(config_path),
    '--output', str(output_root),
    '--mode', mode,
]
print('+', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 6. Visualize results

`bench_aegis.py` writes a timestamped subdirectory under the output root, each containing `benchmark_results.csv` and `benchmark_summary.json`. The cells below pick the most recent run for the selected dataset and produce a small set of standard plots — extend as needed.

In [ ]:
runs = sorted([p for p in output_root.iterdir() if p.is_dir()])
if not runs:
    raise RuntimeError(f'No benchmark runs found under {output_root}')

latest_run = runs[-1]
results_csv = latest_run / 'benchmark_results.csv'
summary_json = latest_run / 'benchmark_summary.json'

print(f'Latest run : {latest_run.relative_to(REPO_ROOT)}')

results = pd.read_csv(results_csv)
with open(summary_json) as f:
    summary = json.load(f)

summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

results['runtime_seconds'].plot.hist(bins=30, ax=axes[0])
axes[0].set_title('Per-file runtime (s)')
axes[0].set_xlabel('seconds')

results['redacted_count'].plot.hist(bins=30, ax=axes[1])
axes[1].set_title('Redacted regions per file')
axes[1].set_xlabel('count')

results['status'].value_counts().plot.bar(ax=axes[2])
axes[2].set_title('Status breakdown')
axes[2].set_ylabel('files')

fig.suptitle(f'{DATASET_KEY} — {len(results)} files')
fig.tight_layout()
plt.show()

### Notes

- **Adding a dataset:** append to `datasets:` in `datasets.yaml`. If its `source.type` is new, register a preparer in section 4's `PREPARERS` dict.
- **Per-dataset benchmark overrides:** add a `benchmark:` block to the dataset entry (e.g. `mode: series`) — `bench_settings` in section 5 merges it over `benchmark_defaults`.
- **Ground-truth scoring:** pass `--ground-truth` through by editing the `cmd` list in section 5; see [`bench_aegis.py`](../../../scripts/bench_aegis.py) for the schema.